In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 64.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import xgboost as xgb
import shap
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
import math

from sklearn.metrics import accuracy_score, f1_score, root_mean_squared_error
from sklearn.datasets import load_diabetes, load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
import sklearn

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")

In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelf from Python file in the drive
!cp /content/drive/MyDrive/...../woodelf.py /content/

import woodelf

# PDP Code

In [ ]:
class CPDVMetric(woodelf.CubeMetric):
    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}
        pdp_values = {}
        if len(s_plus) == 1:
            for f in s_plus:
                pdp_values[f] = 1
        if len(s_plus) == 0:
            for f in s_minus:
                pdp_values[f] = -1
        return pdp_values

In [ ]:
class PathToValuesMatrixLimitSPlus(woodelf.PathToValuesMatrix):
    # Ignored all cubes with |S^+| > MAX_S_PLUS_SIZE to reduce the complexity element of TL3**D to TL2**D*(D**MAX_S_PLUS_SIZE)
    MAX_S_PLUS_SIZE = NotImplemented

    @classmethod
    def map_patterns_to_cube(cls, features_in_path: List[str]):
        updated_wdnf_table = {0: {0: (set(), set())}}
        current_wdnf_table = None
        for feature in features_in_path:
            current_wdnf_table = updated_wdnf_table
            updated_wdnf_table = {}
            for consumer_pattern in current_wdnf_table:
                updated_wdnf_table[consumer_pattern * 2 + 0] = {}
                updated_wdnf_table[consumer_pattern * 2 + 1] = {}
                for background_pattern in current_wdnf_table[consumer_pattern]:
                    s_plus, s_minus = current_wdnf_table[consumer_pattern][background_pattern]

                    # The implementation is identical to the PathToValuesMatrix.map_patterns_to_cube implementation, except for this if.
                    if len(s_plus | {feature}) <= cls.MAX_S_PLUS_SIZE:
                        updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 0] = (s_plus | {feature}, s_minus) # Rule 1

                    updated_wdnf_table[consumer_pattern * 2 + 0][background_pattern * 2 + 1] = (s_plus, s_minus | {feature}) # Rule 2
                    updated_wdnf_table[consumer_pattern * 2 + 1][background_pattern * 2 + 1] = (s_plus, s_minus) # Rule 3

        return updated_wdnf_table

class PathToValuesMatrixLimitSPlusTo1(PathToValuesMatrixLimitSPlus):
    # We uses the fact CPDVMetric ignored all cubes with |S^+| > 1 to reduce the complexity element of TL3**D to TL2**D*D
    MAX_S_PLUS_SIZE = 1

class PathToValuesMatrixLimitSPlusTo2(PathToValuesMatrixLimitSPlus):
    # We uses the fact PDIVOrder1Or2 ignored all cubes with |S^+| > 2 to reduce the complexity element of TL3**D to TL(2**D)*(D**2)
    MAX_S_PLUS_SIZE = 2

In [ ]:
def build_sampled_points_df(data: pd.DataFrame, k: int, seed: int = None):
    """
    Sample k points from every column.
    """
    sample_points_data = {}
    for f in data.columns:
        sample_points_data[f] = list(data[f].sample(k, random_state=seed))
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_equally_distanced_points_df(data: pd.DataFrame, k: int, percentiles: Tuple[float]):
    """
    Take equally distanced points from each column. The min point will be in the precentile percentiles[0]
    and the max point will be in the precentile percentiles[1].
    This is also the default implementation of sklearn
    """
    sample_points_data = {}
    for f in data.columns:
        low, high = np.percentile(data[f].dropna(), [percentiles[0] * 100, percentiles[1]*100])
        # get k equally spaced points between them
        points = np.linspace(low, high, k)
        sample_points_data[f] = list(points)
        sample_points_data[f].sort()
    return pd.DataFrame(sample_points_data)[data.columns]

def build_points_for_full_pdp(data: pd.DataFrame, model, as_df: bool=True):
    """
    Provide the points that will create a full PDP - a graph the will provide the PDV for every x value.
    Does this by collecting all the threshold values from the model. See Sect. of the paper.
    """
    # load the model
    model_objs = woodelf.load_decision_tree_ensamble_model(model, list(data.columns))

    # collect all the theshold values for each feature
    th_values = {f: [] for f in list(data.columns)}
    for tree in model_objs:
        for node in tree.bfs(including_myself=True, including_leaves=False):
            th_values[node.feature_name].append(node.value)

    # Make sure the thesholds are unique and sort them
    for f in th_values:
        th_values[f] = sorted(list(set(th_values[f])))

    if not as_df:
        return th_values

    # zfill
    max_th_length = max([len(thersholds) for thersholds in th_values.values()])
    for f in th_values:
        th_values[f].extend([0] * (max_th_length - len(th_values[f])) )

    # from the built thershold build the points Data Frame
    return pd.DataFrame(th_values)

In [ ]:
def build_points_for_pdp(model, data: pd.DataFrame, k: int = 100, percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose : bool = True):
    start_time = time.time()
    if sampled:
        points_df = build_sampled_points_df(data, k, seed)
    elif full_pdp:
        points_df = build_points_for_full_pdp(data, model)
    else:
        points_df = build_equally_distanced_points_df(data, k, percentiles)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec")
    return points_df

def woodelf_pdp(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=True)
    return woodelf_pdp_given_points_df(model, data, points_df, accurate, centered, GPU), points_df

def woodelf_pdp_given_points_df(model, data: pd.DataFrame, sampled_points_df: pd.DataFrame, accurate: bool = True, centered: bool = True, GPU: bool = False):
    """
    Compute all the PDVs of the provided points. Use WOODELF!
    """
    metric=CPDVMetric()
    p2v = PathToValuesMatrixLimitSPlusTo1(metric)
    if accurate:
        pdvs = woodelf.calculate_background_metric(model, consumer_data=sampled_points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    else:
        pdvs = woodelf.calculate_path_dependent_metric(model, consumer_data=sampled_points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v)
    if centered:
        return pdvs

    avg_prediction = float(model.predict(data).mean())
    for f in pdvs:
        pdvs[f] += avg_prediction
    return pdvs

## Joint DPD code

In [ ]:
# PDP joint

from itertools import combinations

def all_subsets_of_size_0_1_2(s):
    subsets = [set()]
    for k in [1,2]:
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIVOrder1Or2(woodelf.CubeMetric):
    INTERACTION_VALUE = True

    def calc_metric(self, s_plus: Set, s_minus: Set) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets_of_size_0_1_2(s_minus):
            s = tuple(s_plus | sm)
            if len(s) in [1,2]:
                pdivs[s] = (-1) ** (len(sm))
        return pdivs

def bits(n, D):
    bs = []
    for i in range(D):
        bs.append(n % 2)
        n = n // 2
    return reversed(bs)

def build_points_for_joint_pdp(points_df: pd.DataFrame):
    D = math.ceil(math.log2(len(points_df.columns)))
    data = {f: [] for f in points_df.columns}
    k = len(points_df)
    for i, f in enumerate(points_df.columns):
        for b in bits(i, D):
            if b == 0:
                data[f].extend(np.tile(points_df[f].values, k))
            elif b == 1:
                data[f].extend(np.repeat(points_df[f].values, k))
    return pd.DataFrame(data)

def first_different_bit(n1, n2, D):
    assert n1 != n2
    i = 0
    for b1, b2 in zip(bits(n1, D), bits(n2, D)):
        if b1 != b2:
            return i
        i += 1

def clip_result(pdvs, features, k):
    D = math.ceil(math.log2(len(features)))
    feature_to_index = {f:i for i,f in enumerate(features)}
    clipped = {}
    for f1, f2 in pdvs:
        i1 = feature_to_index[f1]
        i2 = feature_to_index[f2]
        h = first_different_bit(i1, i2, D)
        clipped[(f1, f2)] = pdvs[(f1, f2)][h*(k**2): (h+1)*(k**2)]
    return clipped


def woodelf_pdp_joint(model, data: pd.DataFrame, k: int = 100, accurate: bool = True, centered: bool = True, GPU: bool = False,
                percentiles: Tuple[float] = (0.05, 0.95), sampled: bool = False, seed: int = 42, full_pdp: bool = False, verbose: bool = True):
    """
    Compute all the PDVs needed in order to plot the PDP values of all the features. Use WOODELF!
    """
    start_time = time.time()
    original_points_df = build_points_for_pdp(model, data, k, percentiles, sampled, seed, full_pdp, verbose=False)
    if full_pdp:
        k = len(original_points_df)

    points_df = build_points_for_joint_pdp(original_points_df)
    if verbose:
        print(f"Building the points took: {time.time() - start_time} sec. The size of the created df {len(points_df)}")

    metric = PDIVOrder1Or2()
    p2v = PathToValuesMatrixLimitSPlusTo2(metric)
    if accurate:
        pdivs = woodelf.calculate_background_metric(
            model, consumer_data=points_df, background_data=data, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    else:
        pdivs = woodelf.calculate_path_dependent_metric(
            model, consumer_data=points_df, metric=metric, global_importance=False, GPU=GPU, path_to_matrixes_calculator=p2v
        )
    avg_prediction = float(model.predict(data).mean())
    base_pdv = np.array([avg_prediction] * len(points_df))
    zero_array = np.array([0] * len(points_df))
    pdvs = {}

    D = math.ceil(math.log2(len(points_df.columns)))
    points_parts = {f: [points_df[f].values[i:i + k**2] for i in range(0, len(points_df[f]), k**2)] for f in data.columns}
    f1_points = {}
    f2_points = {}
    for i, f1 in enumerate(data.columns):
        for j, f2 in enumerate(data.columns):
            if f1 != f2:
                pair = (f1, f2)
                pdvs[(f1,f2)] = base_pdv + pdivs.get((f1,), zero_array) + pdivs.get((f2,), zero_array) + pdivs.get(pair, zero_array)
                points_part_index = first_different_bit(i,j,D)
                f1_points[(f1,f2)] = points_parts[f1][points_part_index]
                f2_points[(f1,f2)] = points_parts[f2][points_part_index]
    clipped_pdvs = clip_result(pdvs, list(data.columns), k)
    return clipped_pdvs, f1_points, f2_points

## Any Order PDIV code

In [ ]:
from itertools import combinations

def all_subsets(s):
    subsets = []
    for k in range(len(s) + 1):
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIV(woodelf.CubeMetric):
    INTERACTION_VALUE = True
    def calc_metric(
        self, s_plus: Set, s_minus: Set
    ) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets(s_minus):
            s = tuple(s_plus | sm)
            pdivs[s] = (-1) ** (len(sm))
        return pdivs

# Datasets and Models

In [ ]:
from sklearn.datasets import load_diabetes, load_breast_cancer, fetch_california_housing

diabetes = load_diabetes(as_frame=True)
cancer = load_breast_cancer(as_frame=True)
california = fetch_california_housing(as_frame=True)

print("diabetes: ")
print(diabetes.DESCR)
print("cancer: ")
print(cancer.DESCR)
print("california_housing: ")
print(california.DESCR)


df_diabetes = diabetes.frame
df_cancer = cancer.frame
df_california = california.frame

df_diabetes.shape, df_cancer.shape, df_california.shape

diabetes: 
.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variabl

((442, 11), (569, 31), (20640, 9))

In [ ]:
def train_models(dfs, targets, datasets_names, train_ratio=0.8):
    models = {}
    train_sets = {}
    test_sets = {}
    for df, target, name in zip(dfs, targets, datasets_names):
        print()
        X_train, X_test, y_train, y_test = train_test_split(df[[c for c in df.columns if c != target]], df[target], train_size=train_ratio, random_state=42)
        print(f"{name}: Train set size is {len(X_train)}, test set size is {len(X_test)}, the target is {target}, number of columns is {len(X_train.columns)}")
        gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
            max_iter=100, # TODO use 100,
            max_depth=6,
            max_leaf_nodes=None,
            random_state=42
        )
        gradient_boosting_model.fit(X_train, y_train)
        y_pred = gradient_boosting_model.predict(X_test)
        if y_test.nunique() == 2:
            print(f"Accuracy: {accuracy_score(y_test, y_pred.round())}, F1 score: {f1_score(y_test, y_pred.round())}")
        else:
            print(f"RMSE: {root_mean_squared_error(y_test, y_pred)}, RMSE of always taking the mean: {root_mean_squared_error(y_test, pd.Series(y_test.mean(), index=y_test.index))}")

        models[name] = gradient_boosting_model
        train_sets[name] = X_train
        test_sets[name] = X_test

    return models, train_sets, test_sets


In [ ]:
models, trains, tests = train_models([df_diabetes, df_cancer, df_california], ["target", "target", "MedHouseVal"], ["diabetes", "cancer", "california_housing"])


diabetes: Train set size is 353, test set size is 89, the target is target, number of columns is 10
RMSE: 57.04726073307006, RMSE of always taking the mean: 72.78840394263774

cancer: Train set size is 455, test set size is 114, the target is target, number of columns is 30
Accuracy: 0.956140350877193, F1 score: 0.965034965034965

california_housing: Train set size is 16512, test set size is 4128, the target is MedHouseVal, number of columns is 8
RMSE: 0.47841679117287006, RMSE of always taking the mean: 1.1447309632576992


# Running time comparision

In [ ]:
def df_to_latex(df: pd.DataFrame):
    latex = "\\begin{tabularx}{\\columnwidth}{" + "|".join(['r'] * len(df.columns)) + "} \n"
    latex += " & ".join(["\\textbf{" + str(c) + "}" for c in df.columns]) + "\\\\\\hline \n"
    for index, row in df.iterrows():
        latex += " & ".join([f" {v} " for v in row]) + " \\\\ \n"
    latex += "\\end{tabularx}"
    print(latex)

In [ ]:
def woodelf_any_order_pdiv(model, data, global_importance=False, GPU=False):
    return woodelf.calculate_background_metric(
        model, consumer_data=data, background_data=data, metric=PDIV(), global_importance=global_importance, GPU=GPU
    )

def get_woodelf_pdp_testing_params():
    return [
        ("Exact PDP k=5",                  woodelf_pdp, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Exact PDP k=10",                 woodelf_pdp, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Exact PDP k=100",                woodelf_pdp, dict(k = 100,       accurate = True,  centered = False, GPU = False)),
        ("Exact full PDP",                 woodelf_pdp, dict(full_pdp=True, accurate = True,  centered = False, GPU = False)),
        ("Exact Joint PDP k=5",      woodelf_pdp_joint, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Estimated PDP k=5",              woodelf_pdp, dict(k = 5,         accurate = True,  centered = False, GPU = False)),
        ("Estimated Joint PDP k=5",  woodelf_pdp_joint, dict(k = 5,         accurate = False, centered = False, GPU = False)),
        ("Any-Order PDIVs",     woodelf_any_order_pdiv, dict(global_importance = False,                         GPU = False)),
    ]

def messure_woodelf_pdp_times(model, data, params):
    running_results = {}
    for name, woodelf_func, running_params in params:
        print(name + ":")
        start_time = time.time()
        woodelf_func(model, data, **running_params)
        running_results[name] = round(time.time() - start_time, 2)
        print()
    return running_results

def messure_pdp_times_for_many_datasets(models, train_sets, params, alg_name='Woodelf CPU'):
    info = {'Dataset': [], 'Task': [], alg_name: []}
    for name, model in models.items():
        running_results = messure_woodelf_pdp_times(model, train_sets[name], params)
        info['Dataset'].extend([name] * len(running_results))
        info['Task'].extend(list(running_results.keys()))
        info[alg_name].extend(list(running_results.values()))
    return pd.DataFrame(info)

In [ ]:
running_times_df = messure_pdp_times_for_many_datasets(models, trains, get_woodelf_pdp_testing_params())

Exact PDP k=5:
Building the points took: 0.00672602653503418 sec


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 408.98it/s]


cache misses: 124, cache used: 1026


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 941.93it/s]



Exact PDP k=10:
Building the points took: 0.004907131195068359 sec


...

Computing the values: 100%|██████████| 100/100 [00:00<00:00, 150.14it/s]



Any-Order PDIVs:


Preprocessing the trees: 100%|██████████| 100/100 [00:02<00:00, 37.15it/s]


cache misses: 203, cache used: 4287


Computing the values: 100%|██████████| 100/100 [00:03<00:00, 28.00it/s]


In [ ]:
def sota_pdp(model, data, k: int = 5, accurate: bool = True):
    failed_columns = []
    method = 'brute' if accurate else 'recursion'
    # We have to run this one feature at a time, trying to provide significant more than 1 (say 16 features at a time) crashes the RAM :(
    for f in tqdm(data.columns):
        try:
            accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                estimator=model, X=data, features=[f], feature_names=[f], grid_resolution=k, method=method, kind='average'
            )
        except ValueError:
            print(f"{f} has failed...")

    print()
    print(f"{len(failed_columns)} failed columns: {failed_columns}")

def sota_full_pdp(model, data, accurate: bool = True):
    points_for_full_pdp = build_points_for_full_pdp(data=data, model=model, as_df=False)
    failed_columns = []
    method = 'brute' if accurate else 'recursion'
    for f in tqdm(data.columns):
        try:
            if len(points_for_full_pdp[f]) > 0:
                accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                    estimator=model, X=data, features=[f], feature_names=[f], custom_values={f: points_for_full_pdp[f]}, method=method, kind='average'
                )
        except ValueError:
            failed_columns.append(f)

    print()
    print(f"{len(failed_columns)} failed columns are: {failed_columns}")

def sota_joint_pdp(model, data, k: int = 5, accurate: bool = True):
    features = data.columns
    failures = 0
    successes = 0
    method = 'brute' if accurate else 'recursion'
    for i1 in tqdm(range(len(features))):
        for i2 in range(i1 + 1, len(features)):
            try:
                accurate_pdp_result_sklean = sklearn.inspection.partial_dependence(
                    estimator=model, X=data, features=[(i1,i2)], grid_resolution=k, method=method, kind='average'
                )
                successes += 1
            except ValueError:
                failures += 1

    print()
    print(f"Had {failures} failures and {successes} successful computations")

In [ ]:
def get_sota_pdp_testing_params():
    return [
        ("Exact PDP k=5",                  sota_pdp, dict(k = 5,         accurate = True)),
        ("Exact PDP k=10",                 sota_pdp, dict(k = 10,        accurate = True)),
        ("Exact PDP k=100",                sota_pdp, dict(k = 100,       accurate = True)),
        ("Exact full PDP",            sota_full_pdp, dict(               accurate = True)),
        ("Exact Joint PDP k=5",      sota_joint_pdp, dict(k = 5,         accurate = True)),
        ("Estimated PDP k=5",              sota_pdp, dict(k = 5,         accurate = False)),
        ("Estimated Joint PDP k=5",  sota_joint_pdp, dict(k = 5,         accurate = False)),
    ]

In [ ]:
sota_times_df = messure_pdp_times_for_many_datasets(models, trains, get_sota_pdp_testing_params(), alg_name='SOTA')
sota_times_df

Exact PDP k=5:


100%|██████████| 10/10 [00:00<00:00, 40.96it/s]



0 failed columns: []

Exact PDP k=10:


100%|██████████| 10/10 [00:00<00:00, 25.50it/s]



0 failed columns: []

Exact PDP k=100:


...

100%|██████████| 8/8 [00:00<00:00, 323.75it/s]



0 failed columns: []

Estimated Joint PDP k=5:


100%|██████████| 8/8 [00:00<00:00, 68.24it/s]


Had 0 failures and 28 successful computations



,Dataset,Task,SOTA
0,diabetes,Exact PDP k=5,0.25
1,diabetes,Exact PDP k=10,0.40
2,diabetes,Exact PDP k=100,2.73
3,diabetes,Exact full PDP,1.26
4,diabetes,Exact Joint PDP k=5,3.53
5,diabetes,Estimated PDP k=5,0.02
6,diabetes,Estimated Joint PDP k=5,0.10
7,cancer,Exact PDP k=5,0.67
8,cancer,Exact PDP k=10,1.34
9,cancer,Exact PDP k=100,12.19


In [ ]:
def estimate_sota_pdiv_runtime(model, data):
    loaded_model = woodelf.load_decision_tree_ensamble_model(model, list(data.columns))
    total_exponential_time = 0
    n_features_lst = []
    for tree in loaded_model:
        tree_features = [n.feature_name for n in tree.bfs(including_leaves=False)]
        # print(len(tree_features))
        n_features = len(set(tree_features))
        n_features_lst.append(n_features)
        total_exponential_time += 2 ** n_features
    total_exponential_time

    c = 8.66579161428677e-08

    return (c * total_exponential_time * len(data))

pdivs_times = {}
for name, model in models.items():
    pdivs_times[name] = estimate_sota_pdiv_runtime(model, trains[name])
    print(f"{name} Any-Order PDIVs etsimated time: {pdivs_times[name]}")

diabetes Any-Order PDIVs etsimated time: 0.30394466834282335
cancer Any-Order PDIVs etsimated time: 4.3416447903571695
california_housing Any-Order PDIVs etsimated time: 34.982533461510016


In [ ]:
running_times_df["SOTA"] = (
    list(sota_times_df['SOTA'])[0:7] + [round(pdivs_times['diabetes'], 2)] +
    list(sota_times_df['SOTA'])[7:14] + [round(pdivs_times['cancer'], 2)] +
    list(sota_times_df['SOTA'])[14:] + [round(pdivs_times['california_housing'], 2)]
)

In [ ]:
running_times_df

,Dataset,Task,Woodelf CPU,SOTA
0,diabetes,Exact PDP k=5,0.39,0.25
1,diabetes,Exact PDP k=10,0.36,0.40
2,diabetes,Exact PDP k=100,0.40,2.73
3,diabetes,Exact full PDP,0.44,1.26
4,diabetes,Exact Joint PDP k=5,0.72,3.53
5,diabetes,Estimated PDP k=5,0.40,0.02
6,diabetes,Estimated Joint PDP k=5,0.52,0.10
7,diabetes,Any-Order PDIVs,0.92,0.30
8,cancer,Exact PDP k=5,0.40,0.67
9,cancer,Exact PDP k=10,0.39,1.34


In [ ]:
df_to_latex(running_times_df[["Dataset", "Task", "SOTA", "Woodelf CPU"]])

\begin{tabularx}{\columnwidth}{r|r|r|r} 
\textbf{Dataset} & \textbf{Task} & \textbf{SOTA} & \textbf{Woodelf CPU}\\\hline 
 diabetes  &  Exact PDP k=5  &  0.25  &  0.39  \\ 
 diabetes  &  Exact PDP k=10  &  0.4  &  0.36  \\ 
 diabetes  &  Exact PDP k=100  &  2.73  &  0.4  \\ 
 diabetes  &  Exact full PDP  &  1.26  &  0.44  \\ 
 diabetes  &  Exact Joint PDP k=5  &  3.53  &  0.72  \\ 
 diabetes  &  Estimated PDP k=5  &  0.02  &  0.4  \\ 
 diabetes  &  Estimated Joint PDP k=5  &  0.1  &  0.52  \\ 
 diabetes  &  Any-Order PDIVs  &  0.3  &  0.92  \\ 
 cancer  &  Exact PDP k=5  &  0.67  &  0.4  \\ 
 cancer  &  Exact PDP k=10  &  1.34  &  0.39  \\ 
 cancer  &  Exact PDP k=100  &  12.19  &  0.42  \\ 
 cancer  &  Exact full PDP  &  3.06  &  0.45  \\ 
 cancer  &  Exact Joint PDP k=5  &  46.03  &  0.7  \\ 
 cancer  &  Estimated PDP k=5  &  0.05  &  0.45  \\ 
 cancer  &  Estimated Joint PDP k=5  &  1.05  &  0.49  \\ 
 cancer  &  Any-Order PDIVs  &  4.34  &  1.32  \\ 
 california_housing  &  Exact P